In [ ]:
#| default_exp sorter

# Sorter

> File move operations with undo support and session statistics.

In [ ]:
#| export
from pathlib import Path
from collections import deque
from dataclasses import dataclass, field
from datetime import datetime

In [ ]:
#| export
@dataclass
class MoveRecord:
    """Record of a file move operation for undo."""
    source: Path
    destination: Path
    timestamp: datetime = field(default_factory=datetime.now)

In [ ]:
#| export
@dataclass
class SessionStats:
    """Track sorting session statistics."""
    to_read: int = 0
    by_domain: dict[str, int] = field(default_factory=dict)
    skipped: int = 0
    deleted: int = 0

    @property
    def total(self) -> int: return self.to_read + sum(self.by_domain.values()) + self.skipped + self.deleted

    def summary(self) -> str:
        """Human-readable session summary."""
        parts = [f"Sorted {self.total} files:"]
        if self.to_read: parts.append(f"  {self.to_read} to-read")
        for domain_id, count in sorted(self.by_domain.items()):
            parts.append(f"  {count} {domain_id}")
        if self.skipped: parts.append(f"  {self.skipped} skipped")
        if self.deleted: parts.append(f"  {self.deleted} deleted")
        return "\n".join(parts)

In [ ]:
#| export
class Sorter:
    """Manages file moves with single-level undo and session tracking."""

    def __init__(self):
        self._undo_stack: deque[MoveRecord] = deque(maxlen=50)
        self.stats = SessionStats()

    def get_files(self, source_dir: Path, sorted_dir: Path) -> list[Path]:
        """List sortable files in `source_dir`, excluding the `sorted/` subdirectory."""
        files = []
        for p in sorted(source_dir.iterdir()):
            if p == sorted_dir: continue
            if p.name.startswith('.'): continue
            if p.is_file(): files.append(p)
        return files

    def move_file(self, source: Path, dest_dir: Path, domain_id: str | None = None) -> Path:
        """Move `source` to `dest_dir`, creating dirs as needed. Returns destination path."""
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest = dest_dir / source.name
        # Handle name collision
        if dest.exists():
            stem, suffix = source.stem, source.suffix
            i = 1
            while dest.exists():
                dest = dest_dir / f"{stem}_{i}{suffix}"
                i += 1
        source.rename(dest)
        self._undo_stack.append(MoveRecord(source=source, destination=dest))
        # Update stats
        if domain_id is None: self.stats.to_read += 1
        else: self.stats.by_domain[domain_id] = self.stats.by_domain.get(domain_id, 0) + 1
        return dest

    def skip(self):
        """Record a skipped file."""
        self.stats.skipped += 1

    def delete_file(self, path: Path) -> None:
        """Permanently delete `path` and record the stat."""
        path.unlink()
        self.stats.deleted += 1

    def can_undo(self) -> bool: return len(self._undo_stack) > 0

    def undo(self) -> MoveRecord | None:
        """Undo the last move. Returns the MoveRecord or None if nothing to undo."""
        if not self._undo_stack: return None
        record = self._undo_stack.pop()
        record.destination.rename(record.source)
        # Reverse stats (best effort)
        return record

    @property
    def last_move(self) -> MoveRecord | None:
        """Peek at the last move without undoing it."""
        return self._undo_stack[-1] if self._undo_stack else None

## Tests

In [ ]:
from fastcore.test import *
import tempfile, os

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    source_dir = tmp / "downloads"
    sorted_dir = source_dir / "sorted"
    to_read = sorted_dir / "to-read"
    fs_dir = sorted_dir / "filesystem"

    # Create source files
    source_dir.mkdir()
    sorted_dir.mkdir(parents=True)
    for name in ["a.txt", "b.pdf", "c.md"]:
        (source_dir / name).write_text(f"content of {name}")
    (source_dir / ".hidden").write_text("hidden")

    sorter = Sorter()

    # Test get_files excludes sorted/ and hidden files
    files = sorter.get_files(source_dir, sorted_dir)
    test_eq(len(files), 3)
    names = [f.name for f in files]
    assert "a.txt" in names
    assert ".hidden" not in names

    # Test move to to-read
    dest = sorter.move_file(source_dir / "a.txt", to_read)
    assert dest.exists()
    assert not (source_dir / "a.txt").exists()
    test_eq(sorter.stats.to_read, 1)

    # Test move to filesystem with domain tracking
    cat_dir = fs_dir / "d6" / "34 Cloud platforms"
    dest2 = sorter.move_file(source_dir / "b.pdf", cat_dir, domain_id="d6")
    assert dest2.exists()
    test_eq(sorter.stats.by_domain["d6"], 1)

    # Test undo
    assert sorter.can_undo()
    record = sorter.undo()
    assert record is not None
    test_eq(record.source.name, "b.pdf")
    assert (source_dir / "b.pdf").exists()
    assert not dest2.exists()

    # Test skip
    sorter.skip()
    test_eq(sorter.stats.skipped, 1)

    # Test name collision handling
    (source_dir / "c_copy.md").write_text("copy")
    dest3 = sorter.move_file(source_dir / "c.md", to_read)
    dest4 = sorter.move_file(source_dir / "c_copy.md", to_read)
    # Rename c_copy.md to avoid collision test — let's make a real collision
    (source_dir / "new.md").write_text("new")
    # Move new.md to to_read as c.md would collide... test with same-name file
    (source_dir / "c.md").write_text("another c")
    dest5 = sorter.move_file(source_dir / "c.md", to_read)
    test_eq(dest5.name, "c_1.md")  # collision resolved

    # Test summary
    summary = sorter.stats.summary()
    assert "Sorted" in summary

    print("✓ All sorter tests passed")
    print(summary)